## Test Google Drive Export (Colab)

Ce notebook valide la chaine complete:
1. connexion a Google Drive
2. creation d'un dossier de test et de 3 fichiers
3. creation d'un zip
4. telechargement automatique du zip

In [ ]:
from datetime import datetime
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    raise RuntimeError("Ce notebook est prevu pour Colab (google.colab non disponible).")

drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/SailCVExports')
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
TEST_DIR = DRIVE_BASE / f'test_export_{RUN_STAMP}'
TEST_DIR.mkdir(parents=True, exist_ok=True)

print('Drive monte avec succes')
print('Dossier de test:', TEST_DIR)


def auto_download_colab(path_str: str) -> bool:
    try:
        from google.colab import output

        output.eval_js(
            """
            (async () => {
              const p = %s;
              const a = document.createElement('a');
              a.href = '/files/' + encodeURIComponent(p).replace(/%%2F/g, '/');
              a.download = p.split('/').pop();
              document.body.appendChild(a);
              a.click();
              a.remove();
            })();
            """
            % repr(path_str)
        )
        print('Auto-download trigger sent via browser JS.')
        return True
    except Exception as js_err:
        print('JS auto-download failed, fallback to files.download:', js_err)

    try:
        from google.colab import files

        files.download(path_str)
        print('files.download fallback started.')
        return True
    except Exception as dl_err:
        print('Download fallback failed:', dl_err)
        return False

In [ ]:
files_to_create = {
    'readme.txt': 'Test export Sail-CV vers Drive\n',
    'metrics.csv': 'epoch,loss\n1,0.42\n2,0.37\n',
    'config.json': '{"model": "rtdetr", "status": "ok"}\n',
}

for name, content in files_to_create.items():
    (TEST_DIR / name).write_text(content, encoding='utf-8')

print('Fichiers crees:')
for p in sorted(TEST_DIR.iterdir()):
    print('-', p.name)

In [ ]:
import shutil

zip_base = TEST_DIR.parent / f'{TEST_DIR.name}'
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=str(TEST_DIR.parent), base_dir=TEST_DIR.name)

print('ZIP genere:', zip_path)
print(f'Taille: {Path(zip_path).stat().st_size / 1e6:.2f} MB')

In [ ]:
ok = auto_download_colab(zip_path)
if not ok:
    print('Telechargement non demarre automatiquement.')